# RoomRubiksPack Interactive Example

This notebook demonstrates the complete workflow for generating an architectural floorplan layout using the `roomrubikspack` client library.



### 0. Install the Package & Dependencies

We ensure the package is installed along with the optional dependencies required for plotting (`matplotlib`, `networkx`), DXF export (`ezdxf`),.

Choose **Option A** if you are running this notebook in Google Colab or as an end-user. Choose **Option B** if you are running this locally after cloning the GitHub repository.

In [ ]:
# OPTION A: For Google Colab / End-Users (Install latest version from PyPI)
!pip install roomrubikspack --upgrade
!pip install matplotlib networkx ezdxf

# OPTION B: For Local GitHub Developers (Install locally in editable mode)
# Uncomment the lines below if you cloned the repository locally
# !pip install -e ..
# !pip install matplotlib networkx ezdxf


### 1. Initialization and Settings (`rr.init`, `rr.settings`)

We import the package and initialize the session. `rr.init()` clears any previous state.


In [ ]:
import roomrubikspack as rr

# 1. (Optional) Set the measurement unit to meters ('m') or feet ('f')
rr.settings(unit="m")
# Enable Vastu compliance (inherently supported)
rr.vastu(True)


### 2. Grid Control (`rr.constructiongrid`)

We can manipulate the underlying grid module sizes used to generate room dimensions.

In [ ]:
# View or modify the construction grid
rr.constructiongrid(add=5.0)
rr.constructiongrid(remove=9.0)
rr.constructiongrid(reset=True)  # Reset back to defaults
rr.constructiongrid()  # Prints the current grid

### 3. Defining Rooms and Site (`rr.room` and `rr.site`)

Define the rooms by providing explicitly known dimensions (`w` and `h`), or just by providing a target `area` (in square meters). We can also optionally provide a site boundary.

**Important Flags:**
- `startSpace=True`: Indicates the entry room or central hub. Exactly **one** room must have this flag.
- `attachedSpace=True`: Indicates this room is 'dependent' and will be placed *inside* or physically grouped tightly with its connected parent.

In [ ]:
rr.room("living",   "Living Room",  area=20.0, startSpace=True, color="#f2e6d9")
rr.room("kitchen",  "Kitchen",      w=3.0, h=3.0, color="#d9f2d9")
rr.room("bed1",     "Master Bed",   area=16.0, color="#d9d9f2")
rr.room("bath1",    "Attached Bath", area=4.0, attachedSpace=True, color="#e6f2ff")

# (Optional) Define a site boundary
rr.site([{"x": -2, "y": -2}, {"x": 22, "y": -2}, {"x": 22, "y": 22}, {"x": -2, "y": 22}])

### 4. Establishing Connectivity (`rr.connectivity`)

Define which rooms must be physically adjacent to each other. When this runs, it performs a local Euler planarity check to warn you if the graph is impossible to lay out without crossing.

In [ ]:
rr.connectivity(
    ("living", "kitchen"),
    ("living", "bed1"),
    ("bed1",   "bath1")
)

### 5. Visualizing the Graph (`rr.connectivityshow`)

Plots the adjacency graph locally. The start space will be highlighted in green.

In [ ]:
rr.connectivityshow()

### 6. Adding Soft Constraints (`rr.constraint`)

Constraints are sent to the server to guide the Genetic Algorithm. Supported constraints:
- `position`: Bias a room toward a cardinal direction.
- `area`: Attempt to optimize the total building footprint towards a target area.
- `perimeter`: Attempt to minimize the external bounding box perimeter.

In [ ]:
rr.constraint("position", "bed1", "N")
rr.constraint("area", None, 120)
rr.constraint("perimeter", None, "minimize")

### 7. Auto-generating Dimensions (`rr.dimensiongen`)

Sends the rooms to the server to calculate baseline architectural dimensions based on target areas. (Optional, the GA will dynamically explore dimensions within area bounds anyway).

In [ ]:
rr.dimensiongen(avar=0.10, mar=1.5)

### 8. Layout Generation (`rr.generatelayout`)

Sends all session data, connectivity, and constraints to the RoomRubiks backend server to run layout optimization. The engine will dynamically explore different grid-snapped dimension permutations for rooms defined by area to find the tightest fit!

In [ ]:
rr.generatelayout(lvar=0.5, sgap=1.0, max_variations=5)

### 9. Visualizing Layouts Locally (`rr.showlayout`)

Plots the calculated layout variation locally using Matplotlib.

In [ ]:
rr.showlayout(n=1, label=["name", "id", "dim", "area", "vastu"], shownetwork=True)

### 10. Deep Refinement Search (`selv`)

By passing `selv=N`, the engine extracts the exact topological shape of Rank `N` from the previous generation, and performs a deep 45-second optimization strictly locked to that shape. This allows you to explore wide, then deeply exploit a favored layout.

In [ ]:
# DEEP REFINEMENT: Tell the GA to deeply optimize the topological shape of Rank 1
rr.generatelayout(selv=1)

# View the highly-optimized, mathematically clamped variation
rr.showlayout(n=1, label=["name", "id", "dim", "area", "vastu"], shownetwork=True)

### 11. Exporting Layouts Locally (`rr.exportlayout`)

Saves the layout variation locally as JSON or CAD-compatible DXF format.

In [ ]:
rr.exportlayout(n=1, filepath="layout_var1.json")
rr.exportlayout(n=1, filepath="layout_var1.dxf")